# Earlier findings checked on dandiset 000489 version 0.230518.1811

This notebook belongs to the reconciliation of attempt kim-2022-02 with attempt kim-2022-01
(docs/protocol.md, "Repeat attempts"). Of the eight findings of kim-2022-01, six are reported
again by kim-2022-02. The two it did not report, findings 4 and 7, are checked here directly on
[DANDI:000489](https://dandiarchive.org/dandiset/000489/0.230518.1811) version 0.230518.1811,
the version both attempts used. Each section opens with the earlier statement. The data are
streamed with the entry's helper module, `kim2022.py`.

In [ ]:
import re
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import kim2022 as k

pd.set_option("display.width", 200)
assets = k.list_assets("000489")
print("version:", k.VERSIONS["000489"], "| files:", len(assets))

## Earlier finding 4 (format_error)

> Each stitched two-channel image is stored as an ImageSeries with data of shape (22323, 22323, 2)
> and a starting_time rate of 2 Hz. As an ImageSeries the first axis is time, so the file reads as
> 22323 frames of 22323 x 2 pixels; an Images container with two GrayscaleImage objects, or a
> series of shape (1, 22323, 22323) per channel, would describe it. The unit is n.a. and the three
> 25% files share the acquisition name Fig2_25.

The type, shape, rate and unit of each file's acquisition object, and the series name of each
file, read from the file metadata only.

In [ ]:
def series_meta(asset):
    with k.open_nwb("000489", asset["asset_id"]) as f:
        acq = f["acquisition"]
        rows = []
        for name in acq:
            s = acq[name]
            rows.append({
                "subject_id": f["general/subject/subject_id"][()].decode(),
                "session_start": f["session_start_time"][()].decode()[:10],
                "acquisition": name,
                "type": s.attrs.get("neurodata_type"),
                "shape": s["data"].shape,
                "rate_Hz": float(s["starting_time"].attrs["rate"]) if "starting_time" in s else None,
                "unit": s["data"].attrs.get("unit"),
                "session_id": f["general/session_id"][()].decode(),
            })
        return rows

meta = pd.DataFrame([r for a in assets for r in series_meta(a)])
meta.insert(0, "path", [a["path"].split("/")[0] for a in assets])
print(meta.to_string(index=False))

In [ ]:
fig2 = meta[meta.subject_id.str.startswith("Fig2")]
print("acquisition objects per file:", meta.groupby("path").size().unique().tolist())
print("types:", meta.type.value_counts().to_dict())
print("shapes:", meta["shape"].astype(str).value_counts().to_dict())
print("first axis (time for an ImageSeries) length:", sorted(set(s[0] for s in meta["shape"])),
      "| last axis length:", sorted(set(s[-1] for s in meta["shape"])))
print("starting_time rate (Hz):", meta.rate_Hz.value_counts().to_dict(), "| unit:", meta.unit.value_counts().to_dict())
dup = fig2.groupby("acquisition").subject_id.apply(list)
print("acquisition names shared by more than one Fig. 2 file:", dup[dup.apply(len) > 1].to_dict())

The cell above shows whether the layout, rate and unit of the earlier finding are still there.
The conclusion for this finding is in the reconciliation block of `attempts/kim-2022-02.yaml`.

## Earlier finding 7 (selection_undocumented)

> Dish numbers in the subject ids run up to 18 (sub-Fig2-50-18), so more dishes were exposed than
> deposited, and nothing says which dishes form Fig. 2. All unipolar dishes are from 2021-07-29
> and all 75% and 100% dishes from 2021-08-24, a day with no unipolar dish. In these files the 100%
> dishes show more uptake than the unipolar ones (largest mean 761.0 against 571.1 a.u.), which
> does not match Fig. 2C, where every second phase inhibits uptake.

Dish numbers and session dates from the file metadata, the dandiset description, and the YP of
the unipolar and 100% dishes along bb', measured with the helper's ROI (200 um squares from the
midpoint between the electrodes to 3 mm on both sides, the two ROI at each distance averaged per
dish). sub-Fig2-0-03 is left out of the uptake comparison, as its YP channel is the image of
sub-Fig2-25-04 (kim-2022-02, finding 3).

In [ ]:
fig2 = fig2.assign(amp=[int(re.match(r"Fig2_(\d+)_(\d+)", s).group(1)) for s in fig2.subject_id],
                   dish_no=[int(re.match(r"Fig2_(\d+)_(\d+)", s).group(2)) for s in fig2.subject_id])
print(fig2[["subject_id", "amp", "dish_no", "session_start"]].sort_values(["amp", "dish_no"]).to_string(index=False))
print("Fig. 2 files:", len(fig2), "| highest dish number:", fig2.dish_no.max(),
      "| dishes per amplitude:", fig2.groupby("amp").size().to_dict())
print("session dates per amplitude:", fig2.groupby("amp").session_start.apply(lambda s: sorted(set(s))).to_dict())
print("amplitudes per session date:", fig2.groupby("session_start").amp.apply(lambda s: sorted(set(s))).to_dict())
desc = k.version_info("000489")["metadata"]["description"]
print("dandiset description:", desc)
print("description mentions selection, exclusion or dish numbers:",
      {w: (w in desc.lower()) for w in ["select", "exclu", "dish", "replicate", "all experiments"]})

In [ ]:
use = [a for a in assets if re.search(r"sub-Fig2-(0|100)-", a["path"]) and "sub-Fig2-0-03" not in a["path"]]
with ProcessPoolExecutor(3) as ex:
    dishes = list(ex.map(k.process_dish, use))
rows = []
for d in dishes:
    amp = int(re.match(r"Fig2_(\d+)_", d["subject_id"]).group(1))
    for r in d["rois"]:
        rows.append({"dish": d["subject_id"], "amp": amp, "abs_off": round(abs(r["offset_mm"]), 1),
                     "field": r["field_kv_cm"], "YP": r["YP"]})
t = pd.DataFrame(rows).groupby(["amp", "dish", "abs_off"]).agg(YP=("YP", "mean"), field=("field", "mean")).reset_index()
m = t.groupby(["amp", "abs_off"]).agg(mean=("YP", "mean"), sem=("YP", "sem"), field=("field", "mean")).reset_index()
tab = m.pivot(index="abs_off", columns="amp", values="mean")
tab["100% as % of unipolar"] = 100 * tab[100] / tab[0]
tab["field (model, kV/cm)"] = m[m.amp == 0].set_index("abs_off").field
print("dishes:", t.groupby("amp").dish.unique().apply(list).to_dict())
print("mean YP (a.u.) per distance from the centre (mm):")
print(tab.round(1).to_string())
print("largest mean YP: unipolar %.1f, 100%% %.1f a.u." % (tab[0].max(), tab[100].max()))
near = tab.loc[tab.index <= 2.0]
print("ROI within 2 mm where the 100%% mean exceeds the unipolar mean: %d of %d (100%% at %.1f to %.1f%% of unipolar)"
      % ((near[100] > near[0]).sum(), len(near), near["100% as % of unipolar"].min(), near["100% as % of unipolar"].max()))
print("YP per dish and distance (a.u.):")
print(t.pivot_table(index=["amp", "dish"], columns="abs_off", values="YP").round(0).to_string())

The cells above give the dish numbers, dates and uptake on which the reconciliation's
conclusion for this finding rests.